In [ ]:
# Core
import pandas as pd
import numpy as np

# Sklearn - Model & Evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import StandardScaler

# Imbalanced-learn - Resampling
from imblearn.pipeline import Pipeline          # Use this NOT sklearn's Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# read Training Data

train_preprocessed_df = pd.read_csv("../../data/preprocessed/train_preprocessed.csv") 

train_preprocessed_df.head(5)

In [ ]:
corr_matrix = train_preprocessed_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', annot=True, fmt=".2f")

In [ ]:
corr_matrix = corr_matrix.abs()  # abs() catches negative correlations too

upper_triangle = corr_matrix.where(
    mask.astype(bool)
)

to_drop = ['age', 'blood_glucose_level', 'age_bmi_interaction', 'age_hba1c_interaction', 'hypertension']

print(f"Features to drop: {to_drop}")

cleared = train_preprocessed_df.drop(columns=to_drop)
print(f"Remaining features: {cleared.columns.tolist()}")

In [ ]:
X_train = cleared

X_test = pd.read_csv("../../data/preprocessed/validation_preprocessed.csv")

# Align test to train columns exactly
X_test = X_test[X_train.columns]


y_train = X_train['diabetes'].copy()
X_train = X_train.drop('diabetes', axis=1)

y_test = X_test['diabetes'].copy()
X_test  = X_test.drop('diabetes',axis=1 )

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
import pandas as pd

smote = SMOTEENN(random_state=42)
smote_tomek = SMOTETomek(random_state=42)
smote_enn = SMOTEENN(random_state=42)
adasyn = ADASYN(random_state=42)

x_smote, y_smote = smote.fit_resample(X_train, y_train)
x_smote_tomek, y_smote_tomek = smote_tomek.fit_resample(X_train, y_train)
x_smote_enn, y_smote_enn = smote_enn.fit_resample(X_train, y_train)
x_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)

print(pd.DataFrame(x_smote).isna().sum().sum())

pd.concat([pd.DataFrame(x_smote, columns=X_train.columns), pd.Series(y_smote, name='diabetes_target')],  axis=1).to_csv("../../data/imbalance_resolve/train_smote.csv", index=False)
pd.concat([pd.DataFrame(x_smote_tomelink, columns=X_train.columns), pd.Series(y_smote_tomelink, name='diabetes_target')],  axis=1).to_csv("../../data/imbalance_resolve/train_smote_tomek.csv", index=False)
pd.concat([pd.DataFrame(x_smote_enn, columns=X_train.columns), pd.Series(y_smote_enn, name='diabetes_target')],  axis=1).to_csv("../../data/imbalance_resolve/train_smote_enn.csv", index=False)
pd.concat([pd.DataFrame(x_adasyn, columns=X_train.columns), pd.Series(y_adasyn, name='diabetes_target')],  axis=1).to_csv("../../data/imbalance_resolve/ADASYN.csv", index=False)

In [ ]:
baselines = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=42),
}


print("Train features:", X_train.columns.tolist())
print("Test features:", X_test.columns.tolist())

for name, model in baselines.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f"\n{name}")
    print(classification_report(y_test, y_pred))
    print("AUC-ROC:", roc_auc_score(y_test, y_proba))

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

strategies = {
    'SMOTE': SMOTE(random_state=42),
    'SMOTETomek': SMOTETomek(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42),
    'ADASYN': ADASYN(random_state=42)
}

results = []

for name, sampler in strategies.items():
    print(f"\nRunning {name}...")
    
    # Resample ONLY train data
    X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
    
    # Train model
    model = RandomForestClassifier(class_weight='balanced', random_state=42)
    model.fit(X_resampled, y_resampled)
    
    # Evaluate on ORIGINAL test data (never resampled)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred))
    
    # Store key numbers for comparison
    from sklearn.metrics import recall_score, precision_score, f1_score
    results.append({
        'Strategy': name,
        'Recall': recall_score(y_test, y_pred, pos_label=1),
        'Precision': precision_score(y_test, y_pred, pos_label=1),
        'F1': f1_score(y_test, y_pred, pos_label=1),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    })

# Final comparison table
results_df = pd.DataFrame(results)
print("\n===== FINAL COMPARISON =====")
print(results_df.to_string(index=False))

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

strategies = {
    'SMOTE': SMOTE(random_state=42),
    'SMOTETomek': SMOTETomek(random_state=42),
    'SMOTEENN': SMOTEENN(random_state=42),
    'ADASYN': ADASYN(random_state=42)
}

results = []

for name, sampler in strategies.items():
    print(f"\nRunning {name}...")
    
    # Resample ONLY train data
    X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
    
    # Train model
    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(X_resampled, y_resampled)
    
    # Evaluate on ORIGINAL test data (never resampled)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred))
    
    # Store key numbers for comparison
    from sklearn.metrics import recall_score, precision_score, f1_score
    results.append({
        'Strategy': name,
        'Recall': recall_score(y_test, y_pred, pos_label=1),
        'Precision': precision_score(y_test, y_pred, pos_label=1),
        'F1': f1_score(y_test, y_pred, pos_label=1),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    })

# Final comparison table
results_df = pd.DataFrame(results)
print("\n===== FINAL COMPARISON =====")
print(results_df.to_string(index=False))